# 00 — Metadata, media, and cohort freeze

Audits all metadata/media, visualizes the raw cohort, and creates the immutable analysis freeze.

This notebook saves its visual summary and audit tables into separate `figures/` and `tables/` directories. Its final cell states the main output, any decision required, and whether the next stage is allowed.

In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from IPython.display import Image, Markdown, display

def find_project_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "paper1_qc").exists():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the paper_1 project.")

ROOT = find_project_root()
CONFIG = ROOT / "config" / "project.yaml"
OUTPUT = ROOT / "outputs"
MAIN_OUTPUTS = ROOT / "MAIN outputs"
MAIN_OUTPUTS.mkdir(exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

def run_cli(*arguments):
    command = [sys.executable, "-m", "paper1_qc.cli", "--config", str(CONFIG), *arguments]
    print("RUN:", " ".join(map(str, command)))
    subprocess.run(command, cwd=ROOT, check=True)

def read_table(path_without_suffix):
    stem = Path(path_without_suffix)
    parquet = stem.with_suffix(".parquet")
    csv = stem.with_suffix(".csv")
    if parquet.exists():
        return pd.read_parquet(parquet)
    if csv.exists():
        try:
            return pd.read_csv(csv)
        except pd.errors.EmptyDataError:
            return pd.DataFrame()
    raise FileNotFoundError(f"Missing table: {parquet} or {csv}")

def stage_directories(relative_stage):
    stage = OUTPUT / relative_stage
    figures = stage / "figures"
    tables = stage / "tables"
    figures.mkdir(parents=True, exist_ok=True)
    tables.mkdir(parents=True, exist_ok=True)
    return stage, figures, tables

def save_table(frame, directory, name):
    path = Path(directory) / f"{name}.csv"
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False)
    print("TABLE:", path.relative_to(ROOT), f"({len(frame):,} rows)")
    return path

def save_figure(fig, directory, name):
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    png = directory / f"{name}.png"
    svg = directory / f"{name}.svg"
    fig.savefig(png, bbox_inches="tight")
    fig.savefig(svg, bbox_inches="tight")
    print("FIGURE:", png.relative_to(ROOT))
    return png, svg

def stage_gate(stage_name, can_continue, reasons, next_step):
    status = "PASS — safe to continue" if can_continue else "BLOCKED — decision/action required"
    color = "#1B7F3A" if can_continue else "#B22222"
    details = "\n".join(f"- {reason}" for reason in reasons) if reasons else "- No blocking findings."
    display(Markdown(
        f"### {stage_name}: <span style='color:{color}'>{status}</span>\n\n"
        f"{details}\n\n**Next step:** {next_step}"
    ))
    return can_continue

assert CONFIG.exists(), "Copy config/project.example.yaml to config/project.yaml and review it."
print("Project:", ROOT)
print("Config:", CONFIG)
print("MAIN outputs:", MAIN_OUTPUTS)


## Main output and decisions

Main output: the immutable files under `MAIN outputs/00_DATA_FREEZE/<version>/`. The pre-freeze audit is descriptive only. Do not use its candidate/unresolved counts as final cohort counts.

Decision required: every diagnosis not covered by a documented rule must be `ALS`, `CONTROLS`, or `EXCLUDE`. Media must resolve to one decodable path. WAV has priority over WEBM.

In [ ]:
RUN_METADATA_AUDIT = False
RUN_MEDIA_INVENTORY = False  # Slow: probes, fully decodes, and hashes every file.

if RUN_METADATA_AUDIT:
    run_cli("audit")
else:
    print("Metadata audit not rerun.")

if RUN_MEDIA_INVENTORY:
    run_cli("inventory")
else:
    print("Media inventory not rerun. Reuse it only if the data folders have not changed.")

In [ ]:
STAGE, FIGURES, TABLES = stage_directories("00_audit")

def audit_table(stem):
    return read_table(STAGE / stem)

bamboo = audit_table("bamboo_canonical_recordings")
rest = audit_table("rest_canonical_recordings")
bamboo_media = audit_table("bamboo_media_rows_audited")
rest_media = audit_table("rest_media_rows_audited")
bamboo_inventory = audit_table("bamboo_media_inventory")
rest_inventory = audit_table("rest_media_inventory")
exact_pairs = audit_table("exact_bamboo_rest_session_pairs")

order = [
    "Reported ALS",
    "Reported control",
    "Control candidate from ID",
    "Unresolved diagnosis",
    "Conflicting diagnosis",
]

def participant_table(frame, task):
    rows = []
    for subject_id, group in frame.groupby("SubjectID", dropna=False):
        reported = sorted(group["diagnosis_reported"].dropna().astype(str).unique())
        inferred = sorted(group["diagnosis_inferred_from_id"].dropna().astype(str).unique())
        if len(reported) > 1:
            status = "Conflicting diagnosis"
        elif reported == ["ALS"]:
            status = "Reported ALS"
        elif reported == ["CONTROLS"]:
            status = "Reported control"
        elif not reported and "CONTROLS" in inferred:
            status = "Control candidate from ID"
        else:
            status = "Unresolved diagnosis"
        rows.append({
            "SubjectID": subject_id,
            "task": task,
            "cohort_status": status,
            "logical_recordings": group["logical_recording_id"].nunique(),
        })
    return pd.DataFrame(rows)

participants = pd.concat(
    [participant_table(bamboo, "Bamboo"), participant_table(rest, "Rest")],
    ignore_index=True,
)
participant_counts = (
    participants.groupby(["task", "cohort_status"], observed=False)
    .size().rename("participants").reset_index()
)
recording_counts = (
    participants.groupby(["task", "cohort_status"], observed=False)["logical_recordings"]
    .sum().rename("logical_recordings").reset_index()
)
bamboo_ids = set(bamboo["SubjectID"])
rest_ids = set(rest["SubjectID"])
availability = pd.DataFrame({
    "task_availability": ["Both Bamboo and Rest", "Bamboo only", "Rest only"],
    "participants": [
        len(bamboo_ids & rest_ids),
        len(bamboo_ids - rest_ids),
        len(rest_ids - bamboo_ids),
    ],
})
overview = pd.DataFrame([
    {
        "dataset": "Bamboo",
        "metadata_media_rows": len(bamboo_media),
        "physical_files_found": len(bamboo_inventory),
        "logical_recordings": bamboo["logical_recording_id"].nunique(),
        "participants": bamboo["SubjectID"].nunique(),
    },
    {
        "dataset": "Rest",
        "metadata_media_rows": len(rest_media),
        "physical_files_found": len(rest_inventory),
        "logical_recordings": rest["logical_recording_id"].nunique(),
        "participants": rest["SubjectID"].nunique(),
    },
])
pair_summary = pd.DataFrame([{
    "exact_session_pairs_before_freeze": len(exact_pairs),
    "participants_with_pair": exact_pairs["SubjectID"].nunique(),
}])

issue_frames = []
for role in ["bamboo", "rest", "combined"]:
    issues = audit_table(f"{role}_audit_issues")
    if not issues.empty:
        issue_frames.append(issues.assign(dataset=role.capitalize()))
issues = pd.concat(issue_frames, ignore_index=True) if issue_frames else pd.DataFrame()
issue_counts = (
    issues.groupby(["severity", "rule"]).size().rename("issue_rows").reset_index()
    if not issues.empty else pd.DataFrame(columns=["severity", "rule", "issue_rows"])
)

for name, frame in {
    "participant_counts_pre_freeze": participant_counts,
    "logical_recording_counts_pre_freeze": recording_counts,
    "study_overview_pre_freeze": overview,
    "participant_task_availability": availability,
    "exact_session_pair_summary_pre_freeze": pair_summary,
    "metadata_issue_counts": issue_counts,
}.items():
    save_table(frame, TABLES, name)

display(overview)
display(pair_summary)

palette = {
    "Reported ALS": "#C76D6D",
    "Reported control": "#5E81A8",
    "Control candidate from ID": "#E2AE4D",
    "Unresolved diagnosis": "#8C8C8C",
    "Conflicting diagnosis": "#8E5EA2",
}
fig, axes = plt.subplots(2, 2, figsize=(16, 11))
sns.barplot(
    data=participant_counts, x="task", y="participants", hue="cohort_status",
    hue_order=order, palette=palette, ax=axes[0, 0],
)
for container in axes[0, 0].containers:
    axes[0, 0].bar_label(container, fmt="%.0f", padding=3, fontsize=8)
axes[0, 0].set(title="A. Unique participants before diagnosis freeze", xlabel="", ylabel="Participants")

sns.barplot(
    data=recording_counts, x="task", y="logical_recordings", hue="cohort_status",
    hue_order=order, palette=palette, ax=axes[0, 1],
)
for container in axes[0, 1].containers:
    axes[0, 1].bar_label(container, fmt="%.0f", padding=3, fontsize=8)
axes[0, 1].set(title="B. Logical recordings (WAV/WEBM collapsed)", xlabel="", ylabel="Logical recordings")

sns.barplot(
    data=availability, x="participants", y="task_availability",
    hue="task_availability", palette=["#59A14F", "#4C78A8", "#B07AA1"],
    legend=False, ax=axes[1, 0],
)
for container in axes[1, 0].containers:
    axes[1, 0].bar_label(container, fmt="%.0f", padding=4)
axes[1, 0].set(title="C. Participant task availability", xlabel="Participants", ylabel="")

if not issue_counts.empty:
    top = issue_counts.groupby("rule")["issue_rows"].sum().nlargest(10).index
    plot = issue_counts.loc[issue_counts["rule"].isin(top)]
    sns.barplot(
        data=plot, x="issue_rows", y="rule", hue="severity",
        palette={"error": "#D55E5E", "review": "#F2B134"},
        estimator="sum", errorbar=None, ax=axes[1, 1],
    )
    axes[1, 1].set(title="D. Most frequent audit findings", xlabel="Flagged metadata rows", ylabel="")
else:
    axes[1, 1].text(0.5, 0.5, "No audit findings", ha="center", va="center")
fig.suptitle("Paper 1 metadata and cohort audit — pre-freeze", fontsize=17, fontweight="bold")
fig.tight_layout()
save_figure(fig, FIGURES, "metadata_and_cohort_overview_pre_freeze")
plt.show()


## Diagnosis adjudication

In local `config/project.yaml`, list investigator-confirmed exceptional controls under `data_freeze.confirmed_control_subject_ids` and document the evidence. Then run the template cell. Open `config/metadata_adjudication.csv`; complete any remaining row and save it. The freeze cell must remain off until this is done.

In [ ]:
cfg = yaml.safe_load(CONFIG.read_text(encoding="utf-8"))
freeze_cfg = cfg.get("data_freeze", {})
display(pd.DataFrame([{
    "freeze_version": freeze_cfg.get("version"),
    "exact_control_patterns_confirmed": freeze_cfg.get("confirm_configured_control_id_patterns"),
    "exceptional_controls_confirmed": len(freeze_cfg.get("confirmed_control_subject_ids", [])),
    "exceptional_control_evidence_present": bool(freeze_cfg.get("confirmed_control_subject_evidence")),
}]))

run_cli("freeze-template")
adjudication_path = ROOT / freeze_cfg.get(
    "diagnosis_adjudication", "config/metadata_adjudication.csv"
)
adjudication = pd.read_csv(adjudication_path, keep_default_na=False)
display(adjudication)
blank = adjudication["diagnosis_analysis"].astype(str).str.strip().eq("")
diagnosis_ready = stage_gate(
    "Diagnosis adjudication",
    not blank.any(),
    [
        f"{row.SubjectID}: choose ALS, CONTROLS, or EXCLUDE and provide evidence"
        for row in adjudication.loc[blank].itertuples()
    ],
    "Save the completed CSV, rerun this cell, then enable RUN_DATA_FREEZE.",
)

In [ ]:
RUN_DATA_FREEZE = False  # Change to True only when you intend to run this stage.

if RUN_DATA_FREEZE:
    run_cli('freeze')
else:
    print('Freeze not run. Set RUN_DATA_FREEZE=True only after the diagnosis gate passes.')

In [ ]:
cfg = yaml.safe_load(CONFIG.read_text(encoding="utf-8"))
version = str(cfg.get("data_freeze", {}).get("version", "v1"))
FREEZE = MAIN_OUTPUTS / "00_DATA_FREEZE" / version

if (FREEZE / "data_freeze_manifest.json").exists():
    freeze_summary = pd.read_csv(FREEZE / "freeze_summary.csv")
    provenance = pd.read_csv(FREEZE / "diagnosis_provenance.csv")
    bamboo_frozen = pd.read_csv(FREEZE / "frozen_bamboo_recordings.csv")
    rest_frozen = pd.read_csv(FREEZE / "frozen_rest_recordings.csv")
    pairs_frozen = pd.read_csv(FREEZE / "frozen_exact_bamboo_rest_pairs.csv")

    save_table(freeze_summary, TABLES, "frozen_cohort_summary")
    encoding = (
        pd.concat([
            bamboo_frozen.assign(task="Bamboo"),
            rest_frozen.assign(task="Rest"),
        ])
        .groupby(["task", "extension_parsed"]).size()
        .rename("logical_recordings").reset_index()
    )
    save_table(encoding, TABLES, "selected_encoding_counts")

    plot = freeze_summary.loc[
        freeze_summary["diagnosis_analysis"].isin(["ALS", "CONTROLS"])
    ].copy()
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    sns.barplot(data=plot, x="dataset_role", y="participants", hue="diagnosis_analysis", ax=axes[0])
    for container in axes[0].containers:
        axes[0].bar_label(container, fmt="%.0f", padding=3)
    axes[0].set(title="Frozen participants", xlabel="", ylabel="Participants")
    sns.barplot(data=encoding, x="task", y="logical_recordings", hue="extension_parsed", ax=axes[1])
    for container in axes[1].containers:
        axes[1].bar_label(container, fmt="%.0f", padding=3)
    axes[1].set(title="Selected encoding (WAV priority)", xlabel="", ylabel="Logical recordings")
    fig.suptitle("Frozen Paper 1 cohort and selected media")
    fig.tight_layout()
    save_figure(fig, FIGURES, "frozen_cohort_and_encoding_summary")
    plt.show()

    display(freeze_summary)
    freeze_ready = stage_gate(
        "Metadata/media freeze",
        True,
        [],
        "Open the Silero segmentation notebook. All downstream stages now read the frozen tables.",
    )
else:
    freeze_ready = stage_gate(
        "Metadata/media freeze",
        False,
        ["No immutable data_freeze_manifest.json exists for the configured version."],
        "Complete diagnosis adjudication and run the freeze cell.",
    )